In [1]:

import pandas as pd
from opencc import OpenCC
import os

inputdirectory = '../../50 KM Group/Royalties/Statements/Karen/'
outputdirectory = '../../50 KM Group/Royalties/Statements/Karen/_output/'

file_Earth ='Earth/Combined statements/Earth_2022Q4_2025Q4_2_matched_key_columns.csv'
file_Netease = 'netease/combined statements/Netease_statements_2022Q4_2026Q1.xlsx'
file_Rock = 'Rock Music/Consolidated statements/Rock_royalties_2021Q1_2026Q1_2_matched.csv' 
file_TME = 'Tencent - TME/Combined Statements/TME_royalties_2022Q4_2026Q1.xlsx'
file_Sony = 'Sony Music/Consolidated statements/Sony_2018_2026Q1.xlsx'
file_Universal = 'Universal Music/Universal Music Taiwan/Combined statements/Universal_2022Q1_2026Q1.xlsx'
file_Universal_HK = 'Universal Music/Universal Music HK/UMGHK_Royalties_2025Q1_2026Q1.xlsx'
file_Universal_Publishing = 'Universal Publishing/Combined statements/UMP_Royalties_2019Q1_2025H2.xlsx'
file_Douyin = 'Douyin_Bytedance/Douyin_2025Q2.xlsx'
file_Believe = 'Believe/consolidated/Believe_2025Q4_2026Q1.xlsx'

lookup_fx_hkd = 'All labels combined/lookup_tables/global_lookup_cny_hkd.csv'
lookup_fx_ntd = 'All labels combined/lookup_tables/global_lookup_cny_ntd.csv'
lookup_fx_eur = 'All labels combined/lookup_tables/global_lookup_cny_eur.csv'
lookup_song = 'All labels combined/lookup_tables/global_lookup_song_BEST.csv'
lookup_album = 'All labels combined/lookup_tables/global_lookup_album_BEST.csv'
lookup_platform = 'All labels combined/lookup_tables/global_lookup_platform.csv'

outputfile = 'Combined_statements_until_2026Q1_matched.csv'
converter = OpenCC('s2t') 

def readfile(directory,file):
    path = os.path.join(directory, file)
    df = pd.read_csv(path,low_memory=False)
    print(f"The dataframe of the file '{file}' has {df.shape[0]} rows and {len(df.columns)} columns.")
    return df

def readexcel(directory, file,sheetname):
    path = os.path.join(directory, file)
    df = pd.read_excel(path,sheet_name=sheetname)
    print(f"The dataframe '{file}' has {df.shape[0]} rows and {len(df.columns)} columns.")
    return df

def merge(df1, df2, col, what):
    print(f"\nMerging '{what}' on '{col}':")
    empty_cells = df1[col].isna().sum()
    print(f"There are a total of {empty_cells} rows that have no entry in {col}")
    empty_rows = df1[df1[col].isna()]
    print(f"Rows with no entry in '{col}':")
    print(empty_rows)
    print(f"The old dataframe has {df1.shape[0]} rows and {len(df1.columns)} columns.")
    #print(f"Total fee: {df1['Royalties (USD)'].sum()}. Total units: {df1['Units'].sum()}")
    df1.loc[:, col] = df1[col].fillna('XX_UNKNOWN')
    #df1.fillna({col: 'n/a'}, inplace=True)
    df_merged = pd.merge(df1, df2, on=col, how='left')
    #print(f"Total fee: {df_merged['Royalties (USD)'].sum()}. Total units: {df_merged['Units'].sum()}")
    new_columns = df_merged.columns.difference(df1.columns)
    first_new_col = new_columns[0] if not new_columns.empty else None
    empty_cells2 = df_merged[first_new_col].isna().sum() if first_new_col else 0
    diff_empty = empty_cells2 - empty_cells
    if diff_empty == 0:
        print(f"Merging of column {col} was successful")
    else:
        print(f"Merging with issues. There are a total of {diff_empty} cells that could not be matched (see 'match_issues_{what}_{col}.xlsx').")
        empty_rows = df_merged[df_merged[first_new_col].isna()]
        col=col.replace('/','')
        empty_rows.to_excel(f"{outputdirectory}/match_issues_{what}_{col}.xlsx", engine='openpyxl', index=False)
    unused_rows = df2[~df2[col].isin(df_merged[col])]
    if len(unused_rows) > 0:
        unused_rows.to_csv(f'{outputdirectory}/unused_lookup_rows_{what}_{col}.csv', index=False)  
    print(f"The new dataframe has {df_merged.shape[0]} rows and {len(df_merged.columns)} columns.")
    return df_merged 

df_Earth = readfile(inputdirectory,file_Earth)
df_Netease = readexcel(inputdirectory,file_Netease,'data')
df_Rock = readfile(inputdirectory,file_Rock)
df_TME = readexcel(inputdirectory,file_TME,'data')
df_Sony = readexcel(inputdirectory,file_Sony,'data')
df_Universal = readexcel(inputdirectory,file_Universal,'data')
df_Universal_HK = readexcel(inputdirectory,file_Universal_HK,'data')
df_Universal_Publishing = readexcel(inputdirectory,file_Universal_Publishing,'data')
df_Douyin = readexcel(inputdirectory,file_Douyin,'data')
df_Believe = readexcel(inputdirectory,file_Believe,'data')

df_lookup_fx1 = readfile(inputdirectory,lookup_fx_hkd)
df_lookup_fx2 = readfile(inputdirectory,lookup_fx_ntd)
df_lookup_fx3 = readfile(inputdirectory,lookup_fx_eur)
df_lookup_song = readfile(inputdirectory,lookup_song)
df_lookup_album = readfile(inputdirectory,lookup_album)
df_lookup_platform = readfile(inputdirectory,lookup_platform)

The dataframe of the file 'Earth/Combined statements/Earth_2022Q4_2025Q4_2_matched_key_columns.csv' has 1556703 rows and 10 columns.
The dataframe 'netease/combined statements/Netease_statements_2022Q4_2026Q1.xlsx' has 11537 rows and 42 columns.
The dataframe of the file 'Rock Music/Consolidated statements/Rock_royalties_2021Q1_2026Q1_2_matched.csv' has 1442752 rows and 37 columns.
The dataframe 'Tencent - TME/Combined Statements/TME_royalties_2022Q4_2026Q1.xlsx' has 236970 rows and 49 columns.
The dataframe 'Sony Music/Consolidated statements/Sony_2018_2026Q1.xlsx' has 99754 rows and 26 columns.
The dataframe 'Universal Music/Universal Music Taiwan/Combined statements/Universal_2022Q1_2026Q1.xlsx' has 36084 rows and 25 columns.
The dataframe 'Universal Music/Universal Music HK/UMGHK_Royalties_2025Q1_2026Q1.xlsx' has 4649 rows and 27 columns.
The dataframe 'Universal Publishing/Combined statements/UMP_Royalties_2019Q1_2025H2.xlsx' has 45964 rows and 44 columns.
The dataframe 'Douyin_By

In [2]:
# Do Earth df
df_Earth= df_Earth.rename(columns={'Share MABB (CNY)':'Royalty (CNY)'})
df_Earth= df_Earth.rename(columns={'Platform':'Platform_original'})
df_Earth= df_Earth.rename(columns={'Album':'Album_original'})
# df_Earth= df_Earth.rename(columns={'Statement':'Sales_Quarter'})
print(f"Earth: Total fee: {df_Earth['Royalty (CNY)'].sum()}. Total units: {df_Earth['Units'].sum()}")

# Do Netease df
df_Netease= df_Netease.rename(columns={'本月实际分成收益费用 - royalties':'Royalty (CNY)'})
#df_Netease=df_Netease.drop(columns=['Song','歌曲名 - Song title','Album.歌曲名 - Song title'])
#df_Netease= df_Netease.drop(columns=['Song'])
df_Netease= df_Netease.rename(columns={'Song' : 'Song OLD'})
df_Netease= df_Netease.rename(columns={'Total streams and download':'Units'})
df_Netease= df_Netease.rename(columns={'Album':'Album_original'})
df_Netease['Platform_original']= 'Netease'
print(f"Netease: Total fee: {df_Netease['Royalty (CNY)'].sum()}. Total units: {df_Netease['Units'].sum()}")


# Do TME df
df_TME= df_TME.rename(columns={'Fee':'Royalty (CNY)'})
df_TME= df_TME.rename(columns={'Song' : 'Song OLD'})
#df_TME=df_TME.drop(columns=['Song'])
df_TME= df_TME.rename(columns={'Album':'Album_original'})
df_TME= df_TME.rename(columns={'Platform (harmonised)':'Platform_original'})
df_TME= df_TME.rename(columns={'Quarter':'Sales Quarter'})
df_TME['Statement Quarter']=df_TME['Sales Quarter']
print(f"TME: Total fee: {df_TME['Royalty (CNY)'].sum()}. Total units: {df_TME['Units'].sum()}")


# Do Rock df
df_Rock= df_Rock.rename(columns={'Album':'Album_original'})
df_Rock= df_Rock.rename(columns={'Song' : 'Song OLD'})
#df_Rock=df_Rock.drop(columns=['Song'])
df_Rock= df_Rock.rename(columns={'USER':'Platform_original'})
df_Rock= df_Rock.rename(columns={'UNIT':'Units'})
df_Rock['Rev Year'] = df_Rock['Rev Year'].astype(str)
df_Rock['Rev Quarter'] = df_Rock['Rev Quarter'].astype(str)
df_Rock['Sales Quarter'] = df_Rock['Rev Year'].str.cat(df_Rock['Rev Quarter'],sep=' ')
df_Rock['Report year'] = df_Rock['Report year'].astype(str)
df_Rock['Report Quarter'] = df_Rock['Report Quarter'].astype(str)
df_Rock['Statement Quarter'] = df_Rock['Report year'].str.cat(df_Rock['Report Quarter'],sep=' ')
print(f"\nGetting FX right for Rock:")
df_Rock = merge(df_Rock, df_lookup_fx1, 'Sales Quarter','Rock_fx')
df_Rock['Royalty (CNY)']=df_Rock['AMOUNT (HKD)']/df_Rock['FX']
print(f"Rock: Total fee: {df_Rock['Royalty (CNY)'].sum()}. Total units: {df_Rock['Units'].sum()}")


# Do Sony df
df_Sony= df_Sony.rename(columns={'Third Party / DSP (Group)':'Platform_original'})
# df_Sony= df_Sony.rename(columns={'Sales Quarter':'Sales Quarter'})
df_Sony=df_Sony.drop(columns=['Song','Product Title'])
df_Sony= df_Sony.rename(columns={'Pay Units':'Units'})
df_Sony= df_Sony.rename(columns={'Album':'Album_original'})
print(f"\nGetting FX right for Sony:")
df_Sony = merge(df_Sony, df_lookup_fx2, 'Sales Quarter','Sony_fx')
df_Sony['Royalty (CNY)']=df_Sony['Net Amount (NTD)']/df_Sony['FX']
print(f"Sony: Total fee: {df_Sony['Royalty (CNY)'].sum()}. Total units: {df_Sony['Units'].sum()}")


# Do Universal df
df_Universal= df_Universal.rename(columns={'Sub License Description':'Platform_original'})
# df_Universal= df_Universal.rename(columns={'Sales Quarter':'Sales_Quarter'})
#df_Universal= df_Universal.rename(columns={'Int Tune Title':'Song_original'})
df_Universal=df_Universal.drop(columns=['Int Tune Title'])
df_Universal= df_Universal.rename(columns={'Album':'Album_original'})
df_Universal= df_Universal.rename(columns={'Processing Sales':'Units'})
print(f"\nGetting FX right for Universal:")
df_Universal = merge(df_Universal, df_lookup_fx2, 'Sales Quarter','Universal_fx')
df_Universal['Royalty (CNY)']=df_Universal['AIF Rounding']/df_Universal['FX']
print(f"Universal TW: Total fee: {df_Universal['Royalty (CNY)'].sum()}. Total units: {df_Universal['Units'].sum()}")

# Do Universal_HK df
df_Universal_HK= df_Universal_HK.rename(columns={'Platform':'Platform_original'})
# df_Universal= df_Universal.rename(columns={'Sales Quarter':'Sales_Quarter'})
#df_Universal_HK= df_Universal_HK.rename(columns={'Song':'Song_original'})
df_Universal_HK=df_Universal_HK.drop(columns=['Song','Tune Title'])
df_Universal_HK= df_Universal_HK.rename(columns={'Album':'Album_original'})
df_Universal_HK= df_Universal_HK.rename(columns={'Actual ROY-QTY':'Units'})
print(f"\nGetting FX right for Universal_HK:")
df_Universal_HK = merge(df_Universal_HK, df_lookup_fx1, 'Sales Quarter','Universal_HK_fx')
df_Universal_HK['Royalty (CNY)']=df_Universal_HK['Royalty Amount']/df_Universal_HK['FX']
print(f"Universal HK: Total fee: {df_Universal_HK['Royalty (CNY)'].sum()}. Total units: {df_Universal_HK['Units'].sum()}")

# Do Universal_Publishing df
df_Universal_Publishing= df_Universal_Publishing.rename(columns={'Source description Group':'Platform_original'})
df_Universal_Publishing= df_Universal_Publishing.rename(columns={'Song':'Song OLD'})
#df_Universal_Publishing=df_Universal_Publishing.drop(columns=['Song'])
#df_Universal_Publishing= df_Universal_Publishing.rename(columns={'Sales Quarter':'Sales Quarter'})
df_Universal_Publishing= df_Universal_Publishing.rename(columns={'Album':'Album_original'})
df_Universal_Publishing= df_Universal_Publishing.rename(columns={'Statement':'Statement Quarter'})
df_Universal_Publishing["Statement Quarter"] = (
    df_Universal_Publishing["Statement Quarter"]
    .str.replace("H1", "Q2", regex=False)
    .str.replace("H2", "Q4", regex=False)
)

# Do Douyin df
df_Douyin= df_Douyin.rename(columns={'Platform Group':'Platform_original'})
df_Douyin= df_Douyin.rename(columns={'Song':'Song Old'})
df_Douyin= df_Douyin.rename(columns={'Royalty MABB (CNY)':'Royalty (CNY)'})
df_Douyin['Album_original']='爱无所畏'

# Do Believe df
df_Believe = df_Believe.rename(columns={'Platform':'Platform_original'})
df_Believe=df_Believe.drop(columns=['Track title'])
df_Believe=df_Believe.drop(columns=['ISRC'])
df_Believe= df_Believe.rename(columns={'ISRC amended':'ISRC'})
df_Believe= df_Believe.rename(columns={'Release title':'Album_original'})
df_Believe= df_Believe.rename(columns={'Quantity':'Units'})
print(f"\nGetting FX right for Believe:")
df_Believe = merge(df_Believe, df_lookup_fx3, 'Sales Quarter','Believe_fx')
df_Believe['Royalty (CNY)']=df_Believe['Gross Revenue']/df_Believe['FX']
print(f"Believe: Total fee: {df_Believe['Royalty (CNY)'].sum()}. Total units: {df_Believe['Units'].sum()}")

print(f"\nGetting FX right for Universal Publishing:")
df_Universal_Publishing = merge(df_Universal_Publishing, df_lookup_fx1, 'Sales Quarter','UMPG_fx')
df_Universal_Publishing['Royalty (CNY)']=df_Universal_Publishing['Royalties payable']/df_Universal_Publishing['FX']
print(f"Universal Publishing: Total fee: {df_Universal_Publishing['Royalty (CNY)'].sum()}. Total units: {df_Universal_Publishing['Units'].sum()}")


# Creating subsets of relevant columns and merging them
dfs = [df_Earth, df_Netease, df_Rock, df_Sony, df_TME, df_Universal, df_Universal_HK, df_Universal_Publishing, df_Douyin,df_Believe]
sources = ['Earth', 'Netease', 'Rock', 'Sony', 'TME', 'Universal','Universal_HK','UMPG','Douyin','Believe']

columns_to_select = [
    'Statement Quarter',
    'Sales Quarter',
    'Platform_original',
    'ISRC',
    'Album_original',
    'Units',
    'Royalty (CNY)',]

df_subsets = []

for df, source in zip(dfs, sources):
    print(f"\nChecking columns for {source}: {df.columns.tolist()}")
    subset = df.loc[:, columns_to_select].copy()

for df, source in zip(dfs, sources):
    subset = df.loc[:, columns_to_select].copy()
    subset.loc[:, 'Source'] = source 
    print(f"\nThe subset for {source} has {df.shape[0]} rows and {len(df.columns)} columns.")
    empty_cells_p = df['Platform_original'].isna().sum()    
    empty_cells_a = df['Album_original'].isna().sum()
    empty_cells_s = df['ISRC'].isna().sum()
    #empty_cells_in_X = df[df['Platform_original'].isna()]
    #print(empty_cells_in_X)
    print(f"Empty cells in 'Platform_original', 'Album_original', 'ISRC': {empty_cells_p} / {empty_cells_a} / {empty_cells_s}")
    unknown_p = df['Platform_original'].value_counts().get('XX_UNKNOWN', 0)
    unknown_a = df['Album_original'].value_counts().get('XX_UNKNOWN', 0)
    unknown_s = df['ISRC'].value_counts().get('XX_UNKNOWN', 0)
    print(f"Unknown entries in 'Platform_original', 'Album_original', 'ISRC': {unknown_p} / {unknown_a} / {unknown_s}")
    df_subsets.append(subset)

df_combined = pd.concat(df_subsets, ignore_index=True)
print(f"The combined dataframe has {df_combined.shape[0]} rows and {len(df_combined.columns)} columns.")

#df_combined_head = df_combined.head(300)
#df_combined_head.to_excel(f"combined_TEST.xlsx", engine='openpyxl', index=False)




Earth: Total fee: 3082627.625163191. Total units: 354741920.0
Netease: Total fee: 4411781.846706719. Total units: 1316136797
TME: Total fee: 8802156.627865778. Total units: 2455641241.172493

Getting FX right for Rock:

Merging 'Rock_fx' on 'Sales Quarter':
There are a total of 0 rows that have no entry in Sales Quarter
Rows with no entry in 'Sales Quarter':
Empty DataFrame
Columns: [AMOUNT, AMOUNT (HKD), ARTIST, Album_original, Album Type, Amount to Rock (HKD), Amount to Rock (RMB), Base Price, CATALOG NO., CATALOG TITLE, Ctrl.%, Currency, Entry No., FX rate, FX rate_rmb, ISRC, PayType, Payee/Licensor, Payer/Licensee, REVENUE PERIOD, ROYALTY, Release Date, Report Quarter, Report year, Rev Quarter, Rev Year, Royalty Rate%, SHARE AMOUNT (local FX), SONG TITLE, Share%, Song OLD, SongProRata, Territory, Type, Units, Platform_original, WS Price, Sales Quarter, Statement Quarter]
Index: []

[0 rows x 39 columns]
The old dataframe has 1442752 rows and 39 columns.
Merging of column Sales Quar

In [3]:
# Harmonising Songs, Albums and Platforms

#df_combined['Song_original'] = df_combined['Song_original'].astype(str)
#df_combined['Song_original_mod'] = df_combined['Song_original'].str.replace("'", '', regex=False).str.replace(" ", '', regex=False).str.replace("，", '', regex=False).str.replace("\t", '', regex=False).str.replace(',', '', regex=False).str.replace(' ', '', regex=False).str.lower()
#df_combined['Song_original_mod'] = df_combined['Song_original_mod'].apply(converter.convert)

#df_combined['Album_original'] = df_combined['Album_original'].astype(str)
#df_combined['Album_original_mod'] = df_combined['Album_original'].str.replace("'", '', regex=False).str.replace(',', '', regex=False).str.replace(' ', '', regex=False).str.lower()
#df_combined['Album_original_mod'] = df_combined['Album_original_mod'].apply(converter.convert)

df_combined = merge(df_combined, df_lookup_song, 'ISRC','Combined')
print(f"Total fee: {df_combined['Royalty (CNY)'].sum()}. Total units: {df_combined['Units'].sum()}")
df_combined = merge(df_combined, df_lookup_album, 'Album_original','Combined')
print(f"Total fee: {df_combined['Royalty (CNY)'].sum()}. Total units: {df_combined['Units'].sum()}")

df_combined['Album'] = df_combined['Album'].astype(str)
df_combined['Album'] = df_combined['Album'].apply(converter.convert)
#df_combined=df_combined.drop(columns=['Album_original_mod','Song_original_mod'])
#df_combined=df_combined.drop(columns=['Album_original_mod'])

df_combined.fillna({'Platform_original':'XX_UNKNOWN'}, inplace=True)
df_combined = merge(df_combined, df_lookup_platform, 'Platform_original','Combined')
print(f"Total fee: {df_combined['Royalty (CNY)'].sum()}. Total units: {df_combined['Units'].sum()}")

df_combined.fillna({'Country':'XX_UNKNOWN','Platform':'XX_UNKNOWN','Source':'XX_UNKNOWN','Statement':'XX_UNKNOWN',
                    'Song':'XX_UNKNOWN','Album':'XX_UNKNOWN', 'Song (final)':'XX_UNKNOWN', 
                    'Album_type':'XX_UNKNOWN','ISRC (final)':'XX_UNKNOWN'}, inplace=True)
#df_combined.fillna({'Country':'n/a','Platform':'n/a','Source':'n/a','Statement':'n/a','Song':'n/a','Album':'n/a', 'Album_type':'n/a'}, inplace=True)

df_combined = df_combined.sort_index(axis=1)


Merging 'Combined' on 'ISRC':
There are a total of 0 rows that have no entry in ISRC
Rows with no entry in 'ISRC':
Empty DataFrame
Columns: [Statement Quarter, Sales Quarter, Platform_original, ISRC, Album_original, Units, Royalty (CNY), Source]
Index: []
The old dataframe has 3543054 rows and 8 columns.
Merging of column ISRC was successful
The new dataframe has 3543054 rows and 9 columns.
Total fee: 33103276.946174253. Total units: 20368931013.172493

Merging 'Combined' on 'Album_original':
There are a total of 0 rows that have no entry in Album_original
Rows with no entry in 'Album_original':
Empty DataFrame
Columns: [Statement Quarter, Sales Quarter, Platform_original, ISRC, Album_original, Units, Royalty (CNY), Source, Song]
Index: []
The old dataframe has 3543054 rows and 9 columns.
Merging of column Album_original was successful
The new dataframe has 3543054 rows and 11 columns.
Total fee: 33103276.946174253. Total units: 20368931013.172493

Merging 'Combined' on 'Platform_orig

In [4]:
for col in df_combined.columns:
    print(f"Column: {col}")
    print(df_combined[col].map(type).value_counts())
    print()

Column: Album
Album
<class 'str'>    3543054
Name: count, dtype: int64

Column: Album_original
Album_original
<class 'str'>    3543054
Name: count, dtype: int64

Column: Album_type
Album_type
<class 'str'>    3543054
Name: count, dtype: int64

Column: ISRC
ISRC
<class 'str'>    3543054
Name: count, dtype: int64

Column: Platform
Platform
<class 'str'>    3543054
Name: count, dtype: int64

Column: Platform_original
Platform_original
<class 'str'>    3543054
Name: count, dtype: int64

Column: Royalty (CNY)
Royalty (CNY)
<class 'float'>    3543054
Name: count, dtype: int64

Column: Sales Quarter
Sales Quarter
<class 'str'>    3543054
Name: count, dtype: int64

Column: Song
Song
<class 'str'>    3543054
Name: count, dtype: int64

Column: Source
Source
<class 'str'>    3543054
Name: count, dtype: int64

Column: Statement Quarter
Statement Quarter
<class 'str'>    3543054
Name: count, dtype: int64

Column: Units
Units
<class 'float'>    3543054
Name: count, dtype: int64

Column: Unnamed: 2
U

In [5]:
path = os.path.join(outputdirectory, outputfile)
df_combined.to_csv(path, index=False)